将OTU.csv 预处理并转换为lla_compute.py可接受的格式

In [3]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy import stats

# 读取OTU矩阵
log_message(f"读取OTU矩阵: {input_file}")
df = pd.read_csv(input_file)

# 检查并删除无名列
unnamed_cols = [col for col in df.columns if 'Unnamed:' in col]
if unnamed_cols:
    log_message(f"发现无名列: {unnamed_cols}，将被删除")
    df = df.drop(columns=unnamed_cols)

# 检查列名，确认时间点数量
time_points = df.columns[1:].tolist()  # 第一列是OTU ID，后面才是时间点
log_message(f"检测到的时间点列名: {len(time_points)}个")
log_message(f"第一个时间点: {time_points[0]}")
log_message(f"最后一个时间点: {time_points[-1]}")

# 记录原始数据信息
original_otus = df.shape[0]
original_timepoints = len(time_points)  # 使用实际的时间点列数量
log_message(f"原始数据维度: {original_otus} OTUs × {original_timepoints} 时间点")

读取OTU矩阵: /work1/wyx/aaa/elsa/data/otu_matrix.csv
发现无名列: ['Unnamed: 102']，将被删除
检测到的时间点列名: 101个
第一个时间点: 2000/8/17
最后一个时间点: 2010/7/21
原始数据维度: 400 OTUs × 101 时间点


In [ ]:
# otu_matrix.csv已手动筛除
# 1. 筛除全为0的时间点（列）
# 计算每列的和（跳过第一列OTU ID）
column_sums = df.iloc[:, 1:].sum()
non_zero_columns = column_sums[column_sums > 0].index
# 保留非零列和OTU ID列
df_filtered_cols = df.iloc[:, [0]].join(df[non_zero_columns])

# 记录筛选结果
removed_timepoints = original_timepoints - (df_filtered_cols.shape[1] - 1)
log_message(f"移除了 {removed_timepoints} 个全为0的时间点")
log_message(f"筛选后时间点数量: {df_filtered_cols.shape[1] - 1}")

移除了 0 个全为0的时间点
筛选后时间点数量: 101


In [ ]:
# 2. 过滤出现频率过低的OTU
# 计算每个OTU的非零出现次数占总时间点的比例
presence_threshold = 0.2  # 设置出现频率阈值为20%
otu_presence = (df_filtered_cols.iloc[:, 1:] > 0).sum(axis=1) / (df_filtered_cols.shape[1] - 1)
frequent_otus = otu_presence[otu_presence >= presence_threshold].index
df_filtered = df_filtered_cols.iloc[frequent_otus]

# 记录筛选结果
removed_otus = original_otus - df_filtered.shape[0]
log_message(f"使用阈值 {presence_threshold*100}% 过滤低频OTU")
log_message(f"移除了 {removed_otus} 个低频OTU")
log_message(f"筛选后OTU数量: {df_filtered.shape[0]}")


使用阈值 20.0% 过滤低频OTU
移除了 266 个低频OTU
筛选后OTU数量: 134


In [12]:
# 3. 数据标准化处理
# 提取OTU ID和数据部分
otu_ids = df_filtered.iloc[:, 0].values
time_points = df_filtered.columns[1:].tolist()
data = df_filtered.iloc[:, 1:].values

# 检查数据中的缺失值并替换为0
data = np.nan_to_num(data)

In [13]:
# 对数据进行标准化处理
# 方法1: Z-score标准化 (均值为0，标准差为1)
data_zscore = np.zeros_like(data)
for i in range(data.shape[0]):
    row = data[i, :]
    # 只对非零值进行标准化，避免引入噪声
    if np.sum(row > 0) > 1:  # 至少有两个非零值才能计算标准差
        non_zero_mask = row > 0
        if np.sum(non_zero_mask) > 1:
            row_non_zero = row[non_zero_mask]
            mean = np.mean(row_non_zero)
            std = np.std(row_non_zero)
            if std > 0:
                row_non_zero = (row_non_zero - mean) / std
                data_zscore[i, non_zero_mask] = row_non_zero

# 创建输出目录（如果不存在）
output_dir = os.path.dirname(output_file)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 创建输出文件
log_message(f"创建转换后的文件: {output_file}")
with open(output_file, 'w') as f:
    # 写入文件头（以#开头）
    header = '#OTU_ID\t' + '\t'.join(time_points) + '\n'
    f.write(header)
    
    # 写入数据行
    for i, otu_id in enumerate(otu_ids):
        row_data = '\t'.join([str(x) for x in data_zscore[i, :]])
        f.write(f"{otu_id}\t{row_data}\n")

log_message("转换完成！")

创建转换后的文件: /work1/wyx/aaa/elsa/data/otu_matrix_for_lla.txt
转换完成！


In [17]:
# 导入必要的库并设置matplotlib后端
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 查找系统中的中文字体
chinese_fonts = [f.name for f in fm.fontManager.ttflist if 'Noto Sans CJK' in f.name]
if chinese_fonts:
    plt.rcParams['font.sans-serif'] = chinese_fonts
else:
    # 如果没有找到Noto Sans CJK，尝试其他常见中文字体
    for font in ['WenQuanYi Micro Hei', 'Droid Sans Fallback', 'AR PL UMing CN']:
        try:
            fm.findfont(font)
            plt.rcParams['font.sans-serif'] = [font]
            break
        except:
            continue

plt.rcParams['axes.unicode_minus'] = False

# 生成数据分布可视化
plt.figure(figsize=(12, 8))

# 原始数据分布
plt.subplot(2, 1, 1)
plt.hist(data[data > 0].flatten(), bins=50, alpha=0.7)
plt.title('原始数据分布 (非零值)')
plt.xlabel('丰度值')
plt.ylabel('频率')

# 标准化后数据分布
plt.subplot(2, 1, 2)
plt.hist(data_zscore[data_zscore != 0].flatten(), bins=50, alpha=0.7)
plt.title('Z-score标准化后数据分布 (非零值)')
plt.xlabel('标准化丰度值')
plt.ylabel('频率')

plt.tight_layout()
plt.savefig("/work1/wyx/aaa/elsa/data/data_distribution.png", dpi=300)
log_message("数据分布图已保存至: /work1/wyx/aaa/elsa/data/data_distribution.png")

数据分布图已保存至: /work1/wyx/aaa/elsa/data/data_distribution.png


In [ ]:
import pandas as pd
import numpy as np

# 读取原始数据
df = pd.read_csv('otu_matrix.csv')

# 1. 创建时间点文件
dates = df.columns[1:].tolist()  # 获取所有日期(跳过第一列)
timepoints = pd.DataFrame({
    'Index': range(1, len(dates) + 1),  # 从1开始编号
    'Date': dates
})
# 保存时间点文件
timepoints.to_csv('timepoints.txt', sep='\t', index=False)

# 2. 处理数据矩阵
# 重命名列(将日期改为数字序列)
new_columns = ['#'] + [str(i) for i in range(1, len(dates) + 1)]
df.columns = new_columns

# 重命名行(将物种ID改为S1,S2,S3...)
df['#'] = [f'S{i+1}' for i in range(len(df))]

# 保存处理后的数据矩阵
df.to_csv('data.txt', sep='\t', index=False)

print("处理完成！")
print(f"生成了 {len(df)} 个物种的数据")
print(f"时间点数量: {len(dates)}")

处理完成！
生成了 400 个物种的数据
时间点数量: 102


In [5]:
import pandas as pd
import numpy as np

# 读取数据时不把#开头的行当作注释
df = pd.read_csv('otu_matrix_thre20.txt', sep='\t')

# 1. 创建时间点文件
# 获取日期列（跳过第一列OTU_ID）
dates = df.columns[1:].tolist()
timepoints = pd.DataFrame({
    'Index': range(1, len(dates) + 1),
    'Date': dates
})
# 保存时间点文件
timepoints.to_csv('timepoints.txt', sep='\t', index=False)

# 2. 处理数据矩阵
# 重命名列(将日期改为数字序列)
new_columns = ['#'] + [str(i) for i in range(1, len(dates) + 1)]
df.columns = new_columns

# 重命名行(将物种ID改为S1,S2,S3...)
# 注意：这里使用的是原始的OTU_ID列
df['#'] = [f'S{i+1}' for i in range(len(df))]

# 保存处理后的数据矩阵
df.to_csv('data.txt', sep='\t', index=False)

print("处理完成！")
print(f"生成了 {len(df)} 个物种的数据")
print(f"时间点数量: {len(dates)}")

处理完成！
生成了 134 个物种的数据
时间点数量: 101


In [ ]:
# 零值质量检查
#!/usr/bin/env python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

def check_data(data_file):
    """
    检查数据文件的质量和特征
    
    Args:
        data_file: 输入数据文件路径
    """
    print(f"\n检查数据文件: {data_file}")
    print("=" * 50)
    
    # 读取数据
    try:
        # 读取数据，保留第一列作为索引
        data = pd.read_csv(data_file, sep='\t', index_col=0)
        print("\n1. 数据基本信息:")
        print(f"数据维度: {data.shape}")
        print(f"行数(变量数): {data.shape[0]}")
        print(f"列数(时间点): {data.shape[1]}")
        
        # 检查缺失值
        missing_stats = data.isna().sum()
        missing_percent = (missing_stats / len(data)) * 100
        print("\n2. 缺失值统计:")
        print(f"总缺失值数量: {data.isna().sum().sum()}")
        print(f"总缺失值比例: {(data.isna().sum().sum() / data.size) * 100:.2f}%")
        print("\n缺失值最多的前5个变量:")
        print(missing_percent.sort_values(ascending=False).head())
        
        # 检查零值
        zero_stats = (data == 0).sum()
        zero_percent = (zero_stats / len(data)) * 100
        print("\n3. 零值统计:")
        print(f"总零值数量: {(data == 0).sum().sum()}")
        print(f"总零值比例: {((data == 0).sum().sum() / data.size) * 100:.2f}%")
        print("\n零值最多的前5个变量:")
        print(zero_percent.sort_values(ascending=False).head())
        
        # 检查数据分布
        print("\n4. 数据分布统计:")
        print(data.describe())
        
        # 计算每个变量的有效值比例
        valid_percent = ((~data.isna()) & (data != 0)).sum() / len(data) * 100
        print("\n5. 有效值比例统计:")
        print(f"平均有效值比例: {valid_percent.mean():.2f}%")
        print(f"最小有效值比例: {valid_percent.min():.2f}%")
        print(f"最大有效值比例: {valid_percent.max():.2f}%")
        print("\n有效值比例最低的前5个变量:")
        print(valid_percent.sort_values().head())
        
        # 生成可视化图表
        output_dir = "data_quality_plots"
        os.makedirs(output_dir, exist_ok=True)
        
        # 1. 缺失值热图
        plt.figure(figsize=(15, 10))
        sns.heatmap(data.isna(), cmap='YlOrRd', cbar=False)
        plt.title('Missing Values Heatmap')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'missing_values_heatmap.png'))
        plt.close()
        
        # 2. 数据分布箱线图
        plt.figure(figsize=(15, 10))
        data.boxplot()
        plt.title('Data Distribution Boxplot')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'data_distribution_boxplot.png'))
        plt.close()
        
        # 3. 有效值比例条形图
        plt.figure(figsize=(15, 10))
        valid_percent.plot(kind='bar')
        plt.title('Valid Values Percentage by Variable')
        plt.xlabel('Variables')
        plt.ylabel('Valid Values Percentage (%)')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'valid_values_percentage.png'))
        plt.close()
        
        print(f"\n可视化图表已保存到 {output_dir} 目录")
        
    except Exception as e:
        print(f"错误: {str(e)}")
        sys.exit(1)

if __name__ == "__main__":
    if len(sys.argv) != 2:
        print("使用方法: python check_data.py <data_file>")
        sys.exit(1)
    
    data_file = sys.argv[1]
    check_data(data_file)

In [18]:
# 打印使用lla_compute.py的示例命令
log_message("\n使用lla_compute.py分析数据的示例命令:")
log_message("python /work1/wyx/aaa/elsa/lla/lla_compute.py \\")
log_message("    /work1/wyx/aaa/elsa/data/otu_matrix_for_lla.txt \\")
log_message("    /work1/wyx/aaa/elsa/data/lla_results.txt \\")
log_message(f"    -d 0 -p perm -x 1000 -r 1 -s {len(time_points)} \\")
log_message("    -m 0.5 -t simple -f linear -n pnz")


使用lla_compute.py分析数据的示例命令:
python /work1/wyx/aaa/elsa/lla/lla_compute.py \
    /work1/wyx/aaa/elsa/data/otu_matrix_for_lla.txt \
    /work1/wyx/aaa/elsa/data/lla_results.txt \
    -d 0 -p perm -x 1000 -r 1 -s 101 \
    -m 0.5 -t simple -f linear -n pnz
